# Federal urgency-contract tracker (FAR 6.302-2)

When the government skips competition by invoking **unusual and compelling urgency** — the standard that the government would suffer "serious injury, financial or other," without it — it records that in the federal contract data as the *other than full and open competition* reason **`URGENCY (FAR 6.302-2)`**.

This notebook tracks **how much that exception gets used, over time**, straight from the [`abigailhaddad/usaspending-bulk-awards`](https://huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards) mirror of USAspending — queried directly with DuckDB over `hf://`, **no download, no auth**, FY2015–present.

For the **actual justification** — the J&A document agencies must post explaining *why* — see `fetch_justification.py`, which pulls from SAM.gov (needs a free key).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/urgency-tracker/blob/main/demo.ipynb)

In [ ]:
%pip install -q duckdb pandas matplotlib

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# One parquet per fiscal year in the serving layer — read by direct URL (no recursive
# listing, which HuggingFace rate-limits).
SERVE = "https://huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards/resolve/main/serve/contracts"
def year_file(fy):
    return f"read_parquet('{SERVE}/{fy}.parquet')"

# FPDS records the urgency exception in this field.
URGENCY = "other_than_full_and_open_competition ILIKE '%URGENCY%'"

## 1. Urgency usage over time

For each fiscal year: how many contract actions cited urgency, the dollars obligated under them, and urgency's share of *all* non-competed ("other than full and open") dollars.

In [ ]:
rows = []
for fy in range(2015, 2027):
    r = con.execute(f"""
      SELECT
        count(*)                  FILTER (WHERE {URGENCY}) AS urgency_actions,
        sum(federal_action_obligation) FILTER (WHERE {URGENCY}) AS urgency_dollars,
        sum(federal_action_obligation) FILTER (WHERE other_than_full_and_open_competition IS NOT NULL)
                                                                 AS all_noncompete_dollars
      FROM {year_file(fy)}
    """).fetchone()
    rows.append({"fiscal_year": fy, "urgency_actions": r[0] or 0,
                 "urgency_dollars": float(r[1] or 0), "all_noncompete_dollars": float(r[2] or 0)})

trend = pd.DataFrame(rows)
trend["urgency_share"] = trend["urgency_dollars"] / trend["all_noncompete_dollars"]
trend.assign(
    urgency_dollars=lambda d: (d.urgency_dollars / 1e9).round(2).astype(str) + "B",
    urgency_share=lambda d: (d.urgency_share * 100).round(1).astype(str) + "%",
)[["fiscal_year", "urgency_actions", "urgency_dollars", "urgency_share"]]

## 2. The trend chart

Note FY2026 is a **partial** year (through the latest archive snapshot), so its action count is low — but the dollars and share are already elevated.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
plt.rcParams["text.parse_math"] = False
ACCENT = "#69539E"

fys = trend["fiscal_year"].tolist()
dollars = (trend["urgency_dollars"] / 1e9).tolist()
shares = (trend["urgency_share"] * 100).tolist()
partial = [fy == 2026 for fy in fys]

fig, ax = plt.subplots(figsize=(11, 6))
fig.subplots_adjust(top=0.82, bottom=0.10, left=0.08, right=0.93)
bars = ax.bar(range(len(fys)), dollars,
              color=[ACCENT if not p else "#b9a8d6" for p in partial],
              hatch=["" if not p else "//" for p in partial])
for i, (d, s) in enumerate(zip(dollars, shares)):
    ax.text(i, d + max(dollars) * 0.02, f"${d:,.0f}B", ha="center", va="bottom",
            fontsize=9, fontweight="bold", color="#222")

ax.set_xticks(range(len(fys)))
ax.set_xticklabels([f"FY{f}" + ("*" if p else "") for f, p in zip(fys, partial)], fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}B"))
ax.set_ylim(0, max(dollars) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#ededed", lw=0.8); ax.set_axisbelow(True)

# Urgency's share of all non-competed dollars, on a secondary axis.
ax2 = ax.twinx()
ax2.plot(range(len(fys)), shares, color="#E76F51", lw=2, marker="o", ms=4)
ax2.set_ylim(0, max(shares) * 1.3)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}%"))
ax2.spines[["top"]].set_visible(False)
ax2.tick_params(colors="#E76F51"); ax2.set_ylabel("share of non-competed $", color="#E76F51", fontsize=9)

fig.text(0.08, 0.95, "Federal contracting under the urgency exception (FAR 6.302-2)",
         fontsize=15, fontweight="bold", color=ACCENT, ha="left", va="top")
fig.text(0.08, 0.89,
         "Dollars obligated under \u201cunusual and compelling urgency\u201d (bars) and urgency\u2019s share of all "
         "non-competed dollars (line). *FY2026 is a partial year.",
         fontsize=9.5, color="#555", ha="left", va="top")
fig.text(0.08, 0.015,
         "Source: USAspending bulk award archive (U.S. Government, public domain), mirrored at "
         "huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards.",
         fontsize=7.5, color="#999", ha="left", va="bottom")
fig.savefig("urgency_trend.png", dpi=200, facecolor="white")
plt.show()

## 3. Who uses it — by agency

Top agencies by urgency dollars in the most recent (partial) year.

In [ ]:
con.execute(f"""
  SELECT awarding_agency_name AS agency,
         count(*) AS actions,
         round(sum(federal_action_obligation) / 1e6, 1) AS dollars_millions
  FROM {year_file(2026)}
  WHERE {URGENCY}
  GROUP BY 1
  ORDER BY dollars_millions DESC
  LIMIT 12
""").df()

## 4. The biggest urgency awards

Aggregated to the award level (`award_id_piid`) for the most recent year.

In [ ]:
con.execute(f"""
  SELECT award_id_piid AS piid,
         any_value(recipient_name) AS recipient,
         any_value(awarding_sub_agency_name) AS sub_agency,
         round(sum(federal_action_obligation) / 1e6, 1) AS obligated_millions,
         any_value(prime_award_base_transaction_description) AS description
  FROM {year_file(2026)}
  WHERE {URGENCY}
  GROUP BY 1
  ORDER BY obligated_millions DESC
  LIMIT 15
""").df()

## 5. The actual justification (SAM.gov)

USAspending tells you the *reason code* (urgency) — not *why*. The actual **Justification & Approval (J&A)** document, which the agency must post under FAR 6.305, lives on **SAM.gov**.

`fetch_justification.py` searches SAM.gov's opportunities API for the J&A notice tied to a PIID and returns the cited authority and the rationale text. It needs a free `SAM_API_KEY` (from your SAM.gov account):

```bash
SAM_API_KEY=... python fetch_justification.py 70Z02326C93210002
```

(SAM.gov is a separate, keyed source — unlike the HuggingFace mirror above, which is free and needs no auth.)

## Sources & caveats

- **Trend data:** USAspending bulk award archive (FPDS prime contracts), a U.S. Government public-domain work, mirrored at [`abigailhaddad/usaspending-bulk-awards`](https://huggingface.co/datasets/abigailhaddad/usaspending-bulk-awards) and queried with DuckDB over `hf://`.
- **Transaction-level:** rows are contract *actions* (modifications); dollars are `federal_action_obligation` summed (can include de-obligations). An "award" is grouped by `award_id_piid`.
- **FY2026 is partial** — through the latest archive snapshot — so action counts are low even where dollars are high.
- **Urgency** = the `other_than_full_and_open_competition` field containing `URGENCY (FAR 6.302-2)`.
- **Justification text** comes from SAM.gov (see Section 5), a separate source that needs an API key.